In [ ]:
import os
import sys
from dotenv import load_dotenv
import psycopg2

# Загружаем настройки из файла .env
load_dotenv()

def connect_to_db():
    """Создает подключение к базе данных строго через переменные окружения."""
    try:
        connection = psycopg2.connect(
            host=os.getenv('DB_HOST'),
            port=os.getenv('DB_PORT'),
            database=os.getenv('DB_NAME'),
            user=os.getenv('DB_USER'),
            password=os.getenv('DB_PASSWORD')
        )
        connection.autocommit = False
        return connection
    except Exception as error:
        print(f"Ошибка подключения: {error}")
        sys.exit(1)

def init_data_mart_schema(conn):
    """Подготавливаем схему и пустой каркас витрины."""
    queries = """
    CREATE SCHEMA IF NOT EXISTS mart;

    -- Удаляем, если уже есть, чтобы можно было перезапускать скрипт
    DROP TABLE IF EXISTS mart.owner_violations;

    CREATE TABLE mart.owner_violations (
        owner_id INTEGER PRIMARY KEY,
        full_name VARCHAR(255),
        total_fines INTEGER DEFAULT 0,
        total_amount_unpaid NUMERIC(10, 2) DEFAULT 0,
        num_cars INTEGER DEFAULT 0,
        expired_insurances INTEGER DEFAULT 0
    );
    """
    with conn.cursor() as cursor:
        cursor.execute(queries)
    conn.commit()
    print(" Схема и каркас витрины созданы.")

def load_initial_owners_data(conn):
    """ Загружаем базу (всех владельцев). Метрики пока равны 0 (по дефолту)."""
    query = """
    INSERT INTO mart.owner_violations (owner_id, full_name)
    SELECT owner_id, full_name
    FROM owners;
    """
    with conn.cursor() as cursor:
        cursor.execute(query)
    conn.commit()
    print(" Базовые данные владельцев загружены.")

def calculate_num_cars(conn):
    """ Считаем количество автомобилей для каждого владельца."""
    query = """
    WITH CarCounts AS (
        SELECT owner_id, COUNT(car_id) as cnt
        FROM cars
        GROUP BY owner_id
    )
    UPDATE mart.owner_violations AS target
    SET num_cars = source.cnt
    FROM CarCounts AS source
    WHERE target.owner_id = source.owner_id;
    """
    with conn.cursor() as cursor:
        cursor.execute(query)
    conn.commit()
    print("Подсчитано количество автомобилей.")

def calculate_fines_stats(conn):
    """Считаем общее количество штрафов и сумму неоплаченных."""
    query = """
    WITH FineStats AS (
        SELECT 
            c.owner_id,
            COUNT(f.fine_id) as t_fines,
            COALESCE(SUM(CASE WHEN LOWER(f.status) = 'не оплачен' THEN f.amount ELSE 0 END), 0) as t_unpaid
        FROM cars c
        JOIN fines f ON c.car_id = f.car_id
        GROUP BY c.owner_id
    )
    UPDATE mart.owner_violations AS target
    SET 
        total_fines = source.t_fines,
        total_amount_unpaid = source.t_unpaid
    FROM FineStats AS source
    WHERE target.owner_id = source.owner_id;
    """
    with conn.cursor() as cursor:
        cursor.execute(query)
    conn.commit()
    print("Подсчитана статистика по штрафам.")

def calculate_expired_insurances(conn):
    """Шаг 5: Считаем количество просроченных полисов (дата окончания < текущей даты)."""
    query = """
    WITH ExpiredStats AS (
        SELECT 
            c.owner_id,
            COUNT(p.policy_id) as exp_ins
        FROM cars c
        JOIN policies p ON c.car_id = p.car_id
        WHERE p.end_date < CURRENT_DATE
        GROUP BY c.owner_id
    )
    UPDATE mart.owner_violations AS target
    SET expired_insurances = source.exp_ins
    FROM ExpiredStats AS source
    WHERE target.owner_id = source.owner_id;
    """
    with conn.cursor() as cursor:
        cursor.execute(query)
    conn.commit()
    print("Подсчитано количество просроченных страховок.")

def main():
    connection = None
    try:
        # Подключение к БД
        print("Подключение к БД...")
        connection = connect_to_db()
        
        # Запускаем конвейер сборки витрины
        init_data_mart_schema(connection)
        load_initial_owners_data(connection)
        calculate_num_cars(connection)
        calculate_fines_stats(connection)
        calculate_expired_insurances(connection)
        
        print("\n mart.owner_violations собрана")
    except Exception as e:
        print(f"\nОшибка выполнения: {e}")
        if connection:
            connection.rollback()
    finally:
        if connection:
            connection.close()

if __name__ == "__main__":
    main()

Подключение к БД...
[-] Схема и каркас витрины созданы.
[-] Базовые данные владельцев загружены.
[-] Подсчитано количество автомобилей.
[-] Подсчитана статистика по штрафам.
[-] Подсчитано количество просроченных страховок.

Успех! Аналитическая витрина mart.owner_violations собрана!


In [ ]:
import os
import sys
from dotenv import load_dotenv
import psycopg2

# Загружаем настройки из файла .env
load_dotenv()

def connect_to_db():
    """Создает подключение к базе данных строго через переменные окружения."""
    try:
        connection = psycopg2.connect(
            host=os.getenv('DB_HOST'),
            port=os.getenv('DB_PORT'),
            database=os.getenv('DB_NAME'),
            user=os.getenv('DB_USER'),
            password=os.getenv('DB_PASSWORD')
        )
        connection.autocommit = False
        return connection
    except Exception as error:
        print(f"Ошибка подключения: {error}")
        sys.exit(1)

def init_fine_stats_schema(conn):
    """Подготавливаем пустой каркас витрины mart.fine_stats."""
    queries = """
    CREATE SCHEMA IF NOT EXISTS mart;

    DROP TABLE IF EXISTS mart.fine_stats;

    CREATE TABLE mart.fine_stats (
        article VARCHAR(255) PRIMARY KEY,
        total_fines INTEGER DEFAULT 0,
        avg_amount NUMERIC(10, 2) DEFAULT 0,
        payment_rate NUMERIC(5, 2) DEFAULT 0
    );
    """
    with conn.cursor() as cursor:
        cursor.execute(queries)
    conn.commit()
    print(" Каркас витрины fine_stats создан.")

def load_initial_articles(conn):
    """Загружаем уникальные статьи нарушений (каркас)."""
    query = """
    INSERT INTO mart.fine_stats (article)
    SELECT DISTINCT violation 
    FROM fines
    WHERE violation IS NOT NULL;
    """
    with conn.cursor() as cursor:
        cursor.execute(query)
    conn.commit()
    print("Уникальные статьи нарушений загружены.")

def calculate_fines_and_avg(conn):
    """Считаем общее количество штрафов и среднюю сумму по статье."""
    query = """
    WITH Stats AS (
        SELECT 
            violation, 
            COUNT(fine_id) as t_fines, 
            ROUND(AVG(amount), 2) as a_amount
        FROM fines
        GROUP BY violation
    )
    UPDATE mart.fine_stats AS target
    SET 
        total_fines = source.t_fines,
        avg_amount = source.a_amount
    FROM Stats AS source
    WHERE target.article = source.violation;
    """
    with conn.cursor() as cursor:
        cursor.execute(query)
    conn.commit()
    print("Подсчитано количество штрафов и средняя сумма.")

def calculate_payment_rate(conn):
    """Считаем долю оплаченных штрафов (в процентах)."""
    query = """
    WITH PaymentStats AS (
        SELECT 
            violation, 
            COUNT(fine_id) AS total_count,
            -- Считаем только те, у которых статус 'оплачен'
            SUM(CASE WHEN LOWER(status) = 'оплачен' THEN 1 ELSE 0 END) AS paid_count
        FROM fines
        GROUP BY violation
    )
    UPDATE mart.fine_stats AS target
    -- Формула: (оплаченные / все) * 100. Приводим к типу numeric, чтобы не было целочисленного деления
    SET payment_rate = ROUND((source.paid_count::numeric / source.total_count::numeric) * 100, 2)
    FROM PaymentStats AS source
    WHERE target.article = source.violation
      AND source.total_count > 0; -- Защита от деления на ноль
    """
    with conn.cursor() as cursor:
        cursor.execute(query)
    conn.commit()
    print(" Подсчитана доля оплаченных штрафов.")

def main():
    connection = None
    try:
        print("Подключение к БД...")
        connection = connect_to_db()
        
        # Запускаем конвейер сборки второй витрины
        init_fine_stats_schema(connection)
        load_initial_articles(connection)
        calculate_fines_and_avg(connection)
        calculate_payment_rate(connection)
        
        print("\nУспех! Аналитическая витрина mart.fine_stats собрана пошагово!")
    except Exception as e:
        print(f"\nОшибка выполнения: {e}")
        if connection:
            connection.rollback()
    finally:
        if connection:
            connection.close()

if __name__ == "__main__":
    main()

Подключение к БД...
[-] Каркас витрины fine_stats создан.
[-] Уникальные статьи нарушений загружены.
[-] Подсчитано количество штрафов и средняя сумма.
[-] Подсчитана доля оплаченных штрафов.

Успех! Аналитическая витрина mart.fine_stats собрана пошагово!
